In [45]:
from pathlib import Path
import sys
import pandas as pd

import psycopg2
from psycopg2 import sql
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT

from db import get_db_config, fetch_table_df, get_db

In [46]:
conn = get_db()
cur = conn.cursor()

In [13]:
# cur.execute("""commit;""")
# cur.fetchall()

In [15]:
# # delete all data from all tables and reset auto-incrementing ids
# cur.execute("""
# TRUNCATE TABLE 
# questions,
# users,
# subjects,
# topics,
# questions,
# reviews,
# badges,
# stats,
# settings,
# user_subjects,
# user_badges,
# sync_meta
# RESTART IDENTITY CASCADE;
# """)
# cur.execute("""commit;""")

In [16]:
# # schema info
# cur.execute("""
# SELECT
#     conrelid::regclass AS table_name,
#     conname AS constraint_name,
#     pg_get_constraintdef(oid) AS definition
# FROM pg_constraint
# WHERE contype IN ('p','u')
# ORDER BY table_name;
# """)

# tables = cur.fetchall()
# tables

In [74]:
cur.execute(r"""
BEGIN;

UPDATE questions q
SET type = 'english-spell-bee'
FROM topics t
WHERE q.topic_id = t.id
  AND t.key IN (
    'two_letter_words',
    'three_letter_words',
    'four_letter_words',
    'five_letter_words',
    'six_letter_words',
    'seven_letter_words'
  )
  AND q.type = 'spelling-fill-blank';

UPDATE questions
SET question = trim(
  regexp_replace(question, '^\s*Fill in the blank:\s*', '', 'i')
)
WHERE type IN ('english-spell-bee', 'science-spelling', 'spelling-fill-blank');

UPDATE questions
SET question = trim(
  regexp_replace(question, '\m[^[:space:]]*_[^[:space:]]*\M', answer, 'g')
)
WHERE type IN ('english-spell-bee', 'science-spelling', 'spelling-fill-blank')
  AND question ~ '_';

COMMIT;
""")


In [76]:
# import pandas as pd

# # conn.rollback()
# user_id = 3

# overall_sql = """
# SELECT
#   COALESCE(SUM(COALESCE(s.correct, 0) + COALESCE(s.wrong, 0)), 0) AS attempts,
#   COALESCE(SUM(s.correct), 0) AS correct,
#   CASE
#     WHEN COALESCE(SUM(COALESCE(s.correct, 0) + COALESCE(s.wrong, 0)), 0) = 0 THEN 0
#     ELSE ROUND(
#       100.0 * COALESCE(SUM(s.correct), 0) /
#       COALESCE(SUM(COALESCE(s.correct, 0) + COALESCE(s.wrong, 0)), 0)
#     )
#   END AS accuracy
# FROM stats s
# WHERE s.user_id = %s;
# """

# topic_sql = """
# WITH RECURSIVE
# ranked_topics AS (
#   SELECT
#     id,
#     name,
#     subject_id,
#     parent_topic_id,
#     ROW_NUMBER() OVER (
#       PARTITION BY parent_topic_id
#       ORDER BY name
#     ) AS rn
#   FROM topics
# ),
# topic_tree AS (
#   SELECT
#     id,
#     name,
#     subject_id,
#     parent_topic_id,
#     rn,
#     CAST(rn AS TEXT) AS sort_path,
#     0 AS level
#   FROM ranked_topics
#   WHERE parent_topic_id IS NULL
#   UNION ALL
#   SELECT
#     c.id,
#     c.name,
#     c.subject_id,
#     c.parent_topic_id,
#     c.rn,
#     tt.sort_path || '.' || c.rn,
#     tt.level + 1
#   FROM ranked_topics c
#   JOIN topic_tree tt
#     ON c.parent_topic_id = tt.id
# ),
# descendants AS (
#   SELECT id AS topic_id, id AS descendant_id
#   FROM topics
#   UNION ALL
#   SELECT d.topic_id, t.id
#   FROM descendants d
#   JOIN topics t
#     ON t.parent_topic_id = d.descendant_id
# ),
# question_counts AS (
#   SELECT
#     topic_id,
#     COUNT(*) AS total_questions
#   FROM questions
#   GROUP BY topic_id
# ),
# practice_counts AS (
#   SELECT
#     q.topic_id AS topic_id,
#     SUM(COALESCE(s.correct, 0) + COALESCE(s.wrong, 0)) AS practiced,
#     COALESCE(SUM(s.correct), 0) AS correct
#   FROM stats s
#   LEFT JOIN questions q
#     ON q.id::text = s.question_id
#   WHERE s.user_id = %s
#     AND q.topic_id IS NOT NULL
#   GROUP BY q.topic_id
# ),
# topic_totals AS (
#   SELECT
#     d.topic_id,
#     COALESCE(SUM(qc.total_questions), 0) AS total_questions,
#     COALESCE(SUM(pc.practiced), 0) AS practiced,
#     COALESCE(SUM(pc.correct), 0) AS correct
#   FROM descendants d
#   LEFT JOIN question_counts qc
#     ON qc.topic_id = d.descendant_id
#   LEFT JOIN practice_counts pc
#     ON pc.topic_id = d.descendant_id
#   GROUP BY d.topic_id
# )
# SELECT
#   tt.id AS topic_id,
#   tt.name AS topic_name,
#   tt.subject_id AS subject_id,
#   tt.parent_topic_id AS parent_topic_id,
#   tt.level AS level,
#   tt.sort_path AS sort_path,
#   COALESCE(tot.total_questions, 0) AS total_questions,
#   COALESCE(tot.practiced, 0) AS practiced,
#   COALESCE(tot.correct, 0) AS correct,
#   CASE
#     WHEN COALESCE(tot.practiced, 0) = 0 THEN 0
#     ELSE ROUND(100.0 * COALESCE(tot.correct, 0) / COALESCE(tot.practiced, 0))
#   END AS progress
# FROM topic_tree tt
# LEFT JOIN topic_totals tot
#   ON tot.topic_id = tt.id
# ORDER BY tt.sort_path;
# """

# subject_sql = """
# WITH RECURSIVE
# ranked_topics AS (
#   SELECT
#     id,
#     name,
#     subject_id,
#     parent_topic_id,
#     ROW_NUMBER() OVER (
#       PARTITION BY parent_topic_id
#       ORDER BY name
#     ) AS rn
#   FROM topics
# ),
# topic_tree AS (
#   SELECT
#     id,
#     name,
#     subject_id,
#     parent_topic_id,
#     rn,
#     CAST(rn AS TEXT) AS sort_path,
#     0 AS level
#   FROM ranked_topics
#   WHERE parent_topic_id IS NULL
#   UNION ALL
#   SELECT
#     c.id,
#     c.name,
#     c.subject_id,
#     c.parent_topic_id,
#     c.rn,
#     tt.sort_path || '.' || c.rn,
#     tt.level + 1
#   FROM ranked_topics c
#   JOIN topic_tree tt
#     ON c.parent_topic_id = tt.id
# ),
# descendants AS (
#   SELECT id AS topic_id, id AS descendant_id
#   FROM topics
#   UNION ALL
#   SELECT d.topic_id, t.id
#   FROM descendants d
#   JOIN topics t
#     ON t.parent_topic_id = d.descendant_id
# ),
# question_counts AS (
#   SELECT
#     topic_id,
#     COUNT(*) AS total_questions
#   FROM questions
#   GROUP BY topic_id
# ),
# practice_counts AS (
#   SELECT
#     q.topic_id AS topic_id,
#     SUM(COALESCE(s.correct, 0) + COALESCE(s.wrong, 0)) AS practiced,
#     COALESCE(SUM(s.correct), 0) AS correct
#   FROM stats s
#   LEFT JOIN questions q
#     ON q.id::text = s.question_id
#   WHERE s.user_id = %s
#     AND q.topic_id IS NOT NULL
#   GROUP BY q.topic_id
# ),
# topic_totals AS (
#   SELECT
#     d.topic_id,
#     COALESCE(SUM(qc.total_questions), 0) AS total_questions,
#     COALESCE(SUM(pc.practiced), 0) AS practiced,
#     COALESCE(SUM(pc.correct), 0) AS correct
#   FROM descendants d
#   LEFT JOIN question_counts qc
#     ON qc.topic_id = d.descendant_id
#   LEFT JOIN practice_counts pc
#     ON pc.topic_id = d.descendant_id
#   GROUP BY d.topic_id
# ),
# topic_progress AS (
#   SELECT
#     tt.id AS topic_id,
#     tt.name AS topic_name,
#     tt.subject_id AS subject_id,
#     tt.parent_topic_id AS parent_topic_id,
#     COALESCE(tot.total_questions, 0) AS total_questions,
#     COALESCE(tot.practiced, 0) AS practiced,
#     COALESCE(tot.correct, 0) AS correct,
#     CASE
#       WHEN COALESCE(tot.practiced, 0) = 0 THEN 0
#       ELSE ROUND(100.0 * COALESCE(tot.correct, 0) / COALESCE(tot.practiced, 0))
#     END AS progress
#   FROM topic_tree tt
#   LEFT JOIN topic_totals tot
#     ON tot.topic_id = tt.id
# )
# SELECT
#   s.id AS subject_id,
#   s.name AS subject_name,
#   COALESCE(SUM(tp.total_questions), 0) AS total_questions,
#   COALESCE(SUM(tp.practiced), 0) AS practiced,
#   COALESCE(SUM(tp.correct), 0) AS correct,
#   CASE
#     WHEN COALESCE(SUM(tp.practiced), 0) = 0 THEN 0
#     ELSE ROUND(100.0 * COALESCE(SUM(tp.correct), 0) / COALESCE(SUM(tp.practiced), 0))
#   END AS progress
# FROM subjects s
# LEFT JOIN topic_progress tp
#   ON tp.subject_id = s.id
#  AND tp.parent_topic_id IS NULL
# GROUP BY s.id, s.name
# ORDER BY s.name;
# """

# streak_sql = """
# SELECT
#   MAX(CASE WHEN key = 'streak_current_user_3' THEN value END) AS current_streak,
#   MAX(CASE WHEN key = 'streak_longest_user_3' THEN value END) AS longest_streak,
#   MAX(CASE WHEN key = 'streak_last_practice_date_user_3' THEN value END) AS last_practice_date
# FROM settings
# WHERE user_id = %s
#   AND key IN (
#     'streak_current_user_3',
#     'streak_longest_user_3',
#     'streak_last_practice_date_user_3'
#   );
# """

# def fetch_df(title, sql, params, columns=None):
#     cur.execute(sql, params)
#     rows = cur.fetchall()
#     if columns is None:
#         columns = [desc[0] for desc in cur.description]
#     df = pd.DataFrame(rows, columns=columns)
#     print(f"\n{title}")
#     print(df.to_string(index=False))
#     return df

# overall = fetch_df(
#     "Overall card",
#     overall_sql,
#     (user_id,),
#     ["attempts", "correct", "accuracy"],
# )

# topics = fetch_df("Topic breakdown", topic_sql, (user_id,))
# subjects = fetch_df("Subject breakdown", subject_sql, (user_id,))
# streak = fetch_df(
#     "Streak information",
#     streak_sql,
#     (user_id,),
#     ["current_streak", "longest_streak", "last_practice_date"],
# )


In [75]:
# # Sample excute query
# cur.execute("""
# SELECT
#   s.id AS stat_id,
#   s.question_id,
#   s.correct,
#   s.wrong,
#   s.practiced_at,
#   q.id AS question_row_id,
#   q.topic_id,
#   t.name AS topic_name
# FROM stats s
# LEFT JOIN questions q
#   ON q.id::text = s.question_id
# LEFT JOIN topics t
#   ON t.id = q.topic_id
# WHERE s.user_id = 3
# ORDER BY s.practiced_at DESC;


# """)

# tables = cur.fetchall()
# pd.DataFrame(tables)

In [58]:
# query to get all questions that were marked "again" in reviews, along with their subject and topic info

conn.rollback()
cur.execute("""
SELECT
  r.user_id,
  s.name AS subject,
  CASE
    WHEN t.parent_topic_id IS NULL THEN t.name
    ELSE tp.name
  END AS topic_level1,
  CASE
    WHEN t.parent_topic_id IS NULL THEN NULL
    ELSE t.name
  END AS topic_level2,
  q.id AS question_id,
  q.question,
  q.answer,
  r.last_result,
  r.updated_at
FROM reviews r
JOIN questions q
  ON q.id::text = r.question_id
JOIN topics t
  ON t.id = q.topic_id
LEFT JOIN topics tp
  ON tp.id = t.parent_topic_id
JOIN subjects s
  ON s.id = t.subject_id
WHERE r.last_result = 'again'
ORDER BY
  r.user_id,
  s.name,
  topic_level1,
  topic_level2,
  q.id;
""")

pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description])


,user_id,subject,topic_level1,topic_level2,question_id,question,answer,last_result,updated_at
0,1,English,Word Spellings,2 Letter Words,43,The park is by my house.,by,again,2026-04-06 15:19:53.574
1,1,English,Word Spellings,5 Letter Words,144,Mom allows one hour of TV.,allow,again,2026-04-06 15:19:54.090
2,1,English,Word Spellings,7 Letter Words,156,Please arrange the books.,arrange,again,2026-04-06 15:19:54.185
3,1,Science,Science Spellings,Science Long Words,1810,The land was cultivated for crops.,cultivated,again,2026-04-06 15:19:54.759
4,2,English,Word Spellings,4 Letter Words,99,I want some candy.,some,again,2026-04-06 13:07:01.715
5,2,Science,Science Spellings,Science Short Words,1010,I live in the state of Texas.,state,again,2026-04-06 13:08:10.645
6,3,English,Word Spellings,3 Letter Words,117,I was at the park.,was,again,2026-04-08 09:48:40.203
7,3,English,Word Spellings,3 Letter Words,119,This is the way home.,way,again,2026-04-08 09:48:21.933
8,3,English,Word Spellings,3 Letter Words,167,Chocolate bar is my favorite snack.,bar,again,2026-04-08 09:54:38.446
9,3,English,Word Spellings,5 Letter Words,169,We learn basic math.,basic,again,2026-04-08 09:55:05.952


In [28]:
# 1) See what result values actually exist
conn.rollback()
cur.execute("""
SELECT last_result, COUNT(*)
FROM reviews
GROUP BY last_result
ORDER BY COUNT(*) DESC;
""")
pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description])


,last_result,count
0,good,193
1,again,11


In [35]:
# 1) See what result values actually exist
conn.rollback()
cur.execute("""
            SELECT column_name, data_type, is_nullable
FROM information_schema.columns
WHERE table_name = 'questions';
            """)
cur.fetchall()

[('id', 'bigint', 'NO'),
 ('topic_id', 'bigint', 'YES'),
 ('type', 'text', 'YES'),
 ('question', 'text', 'NO'),
 ('answer', 'text', 'NO')]

In [57]:

# 2) Confirm wrong records exist at all (without joins)
conn.rollback()
cur.execute("""
SELECT user_id, question_id, last_result, updated_at
FROM reviews
WHERE last_result IN ('again', 'wrong')
ORDER BY updated_at DESC
LIMIT 100;
""")
pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description])

,user_id,question_id,last_result,updated_at
0,3,169,again,2026-04-08 09:55:05.952
1,3,167,again,2026-04-08 09:54:38.446
2,3,117,again,2026-04-08 09:48:40.203
3,3,119,again,2026-04-08 09:48:21.933
4,1,1810,again,2026-04-06 15:19:54.759
5,1,156,again,2026-04-06 15:19:54.185
6,1,144,again,2026-04-06 15:19:54.090
7,1,43,again,2026-04-06 15:19:53.574
8,2,1010,again,2026-04-06 13:08:10.645
9,2,99,again,2026-04-06 13:07:01.715


#### Wrong answered questions (DND)

In [62]:

conn.rollback()

cur.execute("""
WITH latest_stats AS (
  SELECT DISTINCT ON (s.user_id, s.question_id)
    s.user_id,
    s.question_id,
    s.topic_id,
    s.practiced_at
  FROM stats s
  ORDER BY s.user_id, s.question_id, s.practiced_at DESC
),
wrong_reviews AS (
  SELECT
    r.user_id,
    r.question_id,
    r.last_result,
    r.updated_at
  FROM reviews r
  WHERE r.last_result = 'again'
),
enriched AS (
  SELECT
    wr.user_id,
    u.name AS user_name,
    wr.question_id,
    wr.last_result,
    wr.updated_at,
    COALESCE(ls.topic_id, q.topic_id) AS effective_topic_id,
    q.question AS q_question,
    q.answer   AS q_answer
  FROM wrong_reviews wr
  LEFT JOIN users u
    ON u.id = wr.user_id
  LEFT JOIN latest_stats ls
    ON ls.user_id = wr.user_id
   AND ls.question_id = wr.question_id
  LEFT JOIN questions q
    ON q.id::text = wr.question_id
)
SELECT
  e.user_name,
  s.name AS subject,
  COALESCE(tp.name, t.name) AS topic_level1,
  CASE WHEN tp.id IS NULL THEN NULL ELSE t.name END AS topic_level2,
  e.question_id,
  COALESCE(
    e.q_question,
    CASE
      WHEN e.question_id ~ '^[0-9]+$'
       AND right(e.question_id, 2) = '10'
       AND length(e.question_id) > 2
      THEN
        ((substring(e.question_id from 1 for length(e.question_id)-2))::int)::text
        || ' x 10'
      WHEN e.question_id ~ '^[0-9]+$'
       AND length(e.question_id) > 1
      THEN
        ((substring(e.question_id from 1 for length(e.question_id)-1))::int)::text
        || ' x ' ||
        (right(e.question_id, 1)::int)::text
      ELSE NULL
    END
  ) AS question,
  COALESCE(
    e.q_answer,
    CASE
      WHEN e.question_id ~ '^[0-9]+$'
       AND right(e.question_id, 2) = '10'
       AND length(e.question_id) > 2
      THEN
        (
          (substring(e.question_id from 1 for length(e.question_id)-2))::int
          * 10
        )::text
      WHEN e.question_id ~ '^[0-9]+$'
       AND length(e.question_id) > 1
      THEN
        (
          (substring(e.question_id from 1 for length(e.question_id)-1))::int
          * (right(e.question_id, 1)::int)
        )::text
      ELSE NULL
    END
  ) AS answer,
  e.last_result,
  e.updated_at
FROM enriched e
LEFT JOIN topics t
  ON t.id = e.effective_topic_id
LEFT JOIN topics tp
  ON tp.id = t.parent_topic_id
LEFT JOIN subjects s
  ON s.id = t.subject_id
ORDER BY e.user_name, subject, topic_level1, topic_level2, e.updated_at DESC;
""")

data = cur.fetchall()
pd.DataFrame(data, columns=[desc[0] for desc in cur.description])


,user_name,subject,topic_level1,topic_level2,question_id,question,answer,last_result,updated_at
0,Bhavi,Mathematics,Multiplication Tables,Tables 1-5,43,The park is by my house.,by,again,2026-04-06 15:19:53.574
1,Bhavi,Mathematics,Multiplication Tables,Tables 11-15,156,Please arrange the books.,arrange,again,2026-04-06 15:19:54.185
2,Bhavi,Mathematics,Multiplication Tables,Tables 11-15,144,Mom allows one hour of TV.,allow,again,2026-04-06 15:19:54.090
3,Bhavi,Mathematics,Multiplication Tables,Tables 16-20,1810,The land was cultivated for crops.,cultivated,again,2026-04-06 15:19:54.759
4,Madhu,Mathematics,Multiplication Tables,Tables 6-10,1010,I live in the state of Texas.,state,again,2026-04-06 13:08:10.645
5,Madhu,Mathematics,Multiplication Tables,Tables 6-10,99,I want some candy.,some,again,2026-04-06 13:07:01.715
6,Quiz Kid,Mathematics,Multiplication Tables,Tables 11-15,117,I was at the park.,was,again,2026-04-08 09:48:40.203
7,Quiz Kid,Mathematics,Multiplication Tables,Tables 11-15,119,This is the way home.,way,again,2026-04-08 09:48:21.933
8,Quiz Kid,Mathematics,Multiplication Tables,Tables 16-20,169,We learn basic math.,basic,again,2026-04-08 09:55:05.952
9,Quiz Kid,Mathematics,Multiplication Tables,Tables 16-20,167,Chocolate bar is my favorite snack.,bar,again,2026-04-08 09:54:38.446


In [77]:
# Sample excute query
cur.execute("""
SELECT user_id, key, value, updated_at
FROM settings
ORDER BY user_id, key;

""")

tables = cur.fetchall()
df = pd.DataFrame(tables)
df

,0,1,2,3
0,0,sync_mode,global_on,2026-04-06 04:36:36.000
1,1,active_device_key_user_1,device_bhavi_tab,2026-04-06 03:35:53.608
2,1,device_registry_user_1,"[{""backendKey"":""device_bhavi_tab"",""displayName...",2026-04-06 03:35:53.534
3,1,learn_progress_topic_2,"{""topicId"":2,""cardId"":13,""cardIndex"":2,""totalC...",2026-04-05 06:03:01.915
4,1,practice_session_topic_2,"{""practice"":{""stats"":{""attempts"":0,""correct"":0...",2026-04-05 06:03:07.554
5,2,active_device_key_user_2,device_mabhu_tab,2026-04-06 03:35:53.668
6,2,device_registry_user_2,"[{""backendKey"":""device_bhavi_tab"",""displayName...",2026-04-06 03:35:53.534
7,3,active_device_key_user_3,device_eshu_tablet,2026-04-06 04:17:01.940
8,3,allowed_device_keys_user_3,"[""device_eshu_tablet"",""device_eshu_s22""]",2026-04-05 18:56:15.847
9,3,device_registry_user_3,"[{""backendKey"":""device_bhavi_tab"",""displayName...",2026-04-06 04:36:37.816


In [65]:
# Sample excute query
cur.execute("""
SELECT *
FROM questions
""")

tables = cur.fetchall()
df = pd.DataFrame(tables)

In [66]:
df

,0,1,2,3,4
0,1,6,math-addition,4 + 3 = ?,7
1,2,6,math-addition,9 + 6 = ?,15
2,3,7,math-subtraction,12 - 5 = ?,7
3,4,7,math-subtraction,18 - 9 = ?,9
4,5,8,math-division,24 / 6 = ?,4
...,...,...,...,...,...
3533,1649,22,science-spelling,Every human has a heart.,human
3534,4382,23,science-spelling,The elderly man needs assistance.,elderly
3535,1781,22,science-spelling,I see a bird in the tree.,see
3536,1643,22,science-spelling,I made a paper airplane.,made


In [52]:
# Sample excute query
cur.execute("""
SELECT 
    relname AS table_name,
    n_live_tup AS row_count
FROM pg_stat_user_tables;

""")

tables = cur.fetchall()
pd.DataFrame(tables)

,0,1
0,topics,23
1,badges,0
2,sync_meta,0
3,user_badges,3
4,reviews,0
5,subjects,3
6,settings,12
7,users,3
8,user_subjects,3
9,questions,30


In [ ]:
# table row count
cur.execute("""
SELECT 
    relname AS table_name,
    n_live_tup AS row_count
FROM pg_stat_user_tables;
""")

tables = cur.fetchall()
pd.DataFrame(tables)

,0,1
0,topics,23
1,user_badges,0
2,sync_meta,0
3,badges,0
4,subjects,3
5,user_subjects,3
6,reviews,0
7,users,3
8,stats,0
9,questions,30


In [61]:
# List all tables in the database
cur.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
""")
cur.fetchall()

[('badges',),
 ('subjects',),
 ('topics',),
 ('questions',),
 ('users',),
 ('user_subjects',),
 ('reviews',),
 ('stats',),
 ('user_badges',),
 ('settings',),
 ('sync_meta',)]

In [9]:
fetch_table_df('users', 50)

,id,name,created_at
0,1,Bhavi,2026-03-30 08:53:00.249952
1,2,Madhu,2026-03-30 08:53:00.249952
2,3,Quiz Kid,2026-03-30 08:53:00.249952


In [10]:
fetch_table_df('subjects', 50)

,id,name
0,1,Mathematics
1,2,English
2,3,Science


In [31]:
# IF parent_topic_id is null, then topic is a subtopic, else it's a main topic
fetch_table_df('topics', 100)

,id,subject_id,parent_topic_id,key,name
0,1,1,NaN,multiplication_tables,Multiplication Tables
1,2,1,1.0,tables_1_5,Tables 1-5
2,3,1,1.0,tables_6_10,Tables 6-10
3,4,1,1.0,tables_11_15,Tables 11-15
4,5,1,1.0,tables_16_20,Tables 16-20
5,6,1,NaN,addition,Addition
6,7,1,NaN,subtraction,Subtraction
7,8,1,NaN,division,Division
8,9,1,NaN,fractions,Fractions
9,10,1,NaN,word_problems,Word Problems


In [55]:
df = fetch_table_df('questions', 5000)
df[df['question'].str.contains('fill')]

,id,topic_id,type,question,answer
369,386,14,english-spell-bee,Please fill the glass with water.,fill
1275,1410,23,science-spelling,Mom is filling the glass with water.,filling
2164,2663,23,science-spelling,The cup is filled with milk.,filled
3067,2106,23,science-spelling,I filled out the application for school.,application
3120,1111,23,science-spelling,Balloons expand when filled with air.,expand
3425,1626,22,science-spelling,Please fill the glass with water.,fill


In [34]:
fetch_table_df('reviews', 300)

,user_id,question_id,repetition,interval,ease_factor,next_review,last_result,rev_id,last_modified_rev,sync_version,updated_at
0,3,19,0,0,2.5,1775469643755,again,1775469039021,1775469039021,1,2026-04-06 15:20:39.021
1,3,17,1,1,2.5,1775555484206,good,1775469084274,1775469084274,1,2026-04-06 15:21:24.274
2,3,13,1,1,2.5,1775584260168,good,1775497860199,1775497860199,15,2026-04-06 23:21:00.199
3,3,14,0,0,2.5,1775498474555,again,1775497869792,1775497869792,14,2026-04-06 23:21:09.792
4,3,112,1,1,2.5,1775514969902,good,1775428569957,1775428569957,3,2026-04-06 04:36:46.848
...,...,...,...,...,...,...,...,...,...,...,...
199,1,201,1,1,2.5,1775548983248,good,1775462583265,1775462583265,2,2026-04-06 15:19:54.928
200,1,208,1,1,2.5,1775548990739,good,1775462590758,1775462590758,2,2026-04-06 15:19:54.935
201,1,209,1,1,2.5,1775548996564,good,1775462596582,1775462596582,2,2026-04-06 15:19:54.943
202,1,203,1,1,2.5,1775549001315,good,1775462601348,1775462601348,2,2026-04-06 15:19:54.952


In [36]:
fetch_table_df('topics', 50)

,id,subject_id,parent_topic_id,key,name
0,1,1,NaN,multiplication_tables,Multiplication Tables
1,2,1,1.0,tables_1_5,Tables 1-5
2,3,1,1.0,tables_6_10,Tables 6-10
3,4,1,1.0,tables_11_15,Tables 11-15
4,5,1,1.0,tables_16_20,Tables 16-20
5,6,1,NaN,addition,Addition
6,7,1,NaN,subtraction,Subtraction
7,8,1,NaN,division,Division
8,9,1,NaN,fractions,Fractions
9,10,1,NaN,word_problems,Word Problems


In [14]:
fetch_table_df('stats', 500)

""


In [15]:
fetch_table_df('settings', 10)

""


In [16]:
fetch_table_df('user_subjects', 10)

,user_id,subject_id
0,1,1
1,2,1
2,3,1


In [17]:
fetch_table_df('user_badges', 5)

""


In [69]:
df = fetch_table_df('questions', 10000)
df

,id,topic_id,type,question,answer
0,1,6,math-addition,4 + 3 = ?,7
1,2,6,math-addition,9 + 6 = ?,15
2,3,7,math-subtraction,12 - 5 = ?,7
3,4,7,math-subtraction,18 - 9 = ?,9
4,5,8,math-division,24 / 6 = ?,4
...,...,...,...,...,...
3533,1649,22,science-spelling,Every human has a heart.,human
3534,4382,23,science-spelling,The elderly man needs assistance.,elderly
3535,1781,22,science-spelling,I see a bird in the tree.,see
3536,1643,22,science-spelling,I made a paper airplane.,made


In [71]:
df[df['question'].str.contains('Fill')]

,id,topic_id,type,question,answer
3254,3053,23,science-spelling,Fill the bottle with water.,bottle
